# Llama3 Base Model Evaluation

## Import Libraries

In [1]:
!pip install -q --upgrade bitsandbytes

In [2]:
!wget -q https://raw.githubusercontent.com/KumudithaSilva/llama3-domain-adaptation/feature-base-model/evaluator.py -O evaluator.py

In [3]:
import os
import re
import math
from tqdm import tqdm
from google.colab import userdata
from huggingface_hub import login
from datasets import load_dataset, Dataset, DatasetDict
import torch
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments, set_seed
from peft import LoraConfig, PeftConfig
from datetime import  datetime
from evaluator import evaluate

## Load Dataset From HuggingFace

In [4]:
BASE_MODEL = "meta-llama/Llama-3.2-3B"

PROJECT_NAME = "stream_price"

RUN_NAME =  f"{datetime.now():%Y-%m-%d_%H.%M.%S}"
PROJECT_RUN_NAME = f"{PROJECT_NAME}-{RUN_NAME}"

DATA_USER = "KumudithaSilva"
DATASET_NAME = f"{DATA_USER}/stream_items_prompt_lite"

In [5]:
hf_token = userdata.get('HUGGING_KEY')
login(hf_token)

In [6]:
dataset = load_dataset(DATASET_NAME)

train = dataset['train'].remove_columns(['id'])
val = dataset['validation'].remove_columns(['id'])
test = dataset['test'].remove_columns(['id'])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [7]:
train[0]

{'prompt': 'How much this stream game cost to the nearest dollar?\n\n\nGame: FaceValue\nPeak CCU: 0\nRequired Age: 0\nDLC Count: 0\nSupports Windows: True\nSupports Mac: False\nSupports Linux: False\nPositive Reviews: 7\nNegative Reviews: 0\nAchievements: 9\nRecommendations: 0\nRelease Date: 2023-2-21\nEstimated Owners: 50000 - 100000\nLanguages Supported: 1\nDevelopers: 1\nPublishers: 1\nCategories: 2\nGenres: 2\nDescription: Explore dice puzzles in a tranquil isometric world.\n\n\nPrice is $',
 'completion': '2.00'}

## Load Llama Model

In [8]:
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
    )

## Llama Model

In [9]:
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=quant_config,
    device_map="auto",
    )

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

## Tokenizer

In [10]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

In [11]:
base_model.generation_config.pad_token_id = tokenizer.pad_token_id

## Memory Footprint

In [12]:
print(f"Memory footprint: {base_model.get_memory_footprint() / 1e9:.1f} GB")

Memory footprint: 2.2 GB


## Model Prediction

In [13]:
def model_predict(item):
    inputs = tokenizer(item["prompt"],return_tensors="pt").to("cuda")

    with torch.no_grad():
        output_ids = base_model.generate(**inputs, max_new_tokens=8)

    prompt_len = inputs["input_ids"].shape[1]
    generated_ids = output_ids[0, prompt_len:]

    return tokenizer.decode(generated_ids)

In [14]:
test[0]

{'prompt': 'How much this stream game cost to the nearest dollar?\n\n\nGame: Rocket Explorer\nPeak CCU: 0\nRequired Age: 0\nDLC Count: 0\nSupports Windows: True\nSupports Mac: True\nSupports Linux: True\nPositive Reviews: 16\nNegative Reviews: 11\nAchievements: 3\nRecommendations: 0\nRelease Date: 2021-8-21\nEstimated Owners: 0 - 20000\nLanguages Supported: 1\nDevelopers: 1\nPublishers: 1\nCategories: 4\nGenres: 4\nDescription: Explore and interact with rockets in VR.\n\n\nPrice is $',
 'completion': '12.99'}

## Evaluation

In [15]:
evaluate(model_predict, test)